# Task 3 — TinyConvNeXt and TinyHRNet E7 Experiments

This notebook trains only the two frozen E7 architecture children. **Run All starts ten new fits:** five Usage TinyConvNeXt-18 folds first, then five Gender TinyHRNet-20 folds. It never retrains E1–E6.

## 1. Mount Drive and load the submitted branch

Drive supplies the teacher data, accepted parents, persistent registry, and E7 artifacts.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import zipfile

REPO_URL = "https://github.com/TrnLin/MLA2.git"
BRANCH = "task-3-gender-usage-classification"
REPO_DIR = Path("/content/MLA2")
DRIVE_MOUNT = Path("/content/drive")
DRIVE_PROJECT_DIR = DRIVE_MOUNT / "MyDrive/MLA2"
DATA_ZIP = DRIVE_PROJECT_DIR / "data/task3-data.zip"
DRIVE_TASK_DIR = DRIVE_PROJECT_DIR / "task3"
DRIVE_REGISTRY = DRIVE_TASK_DIR / "results/runs.csv"


def run_checked(command, *, cwd=None):
    command = [str(part) for part in command]
    print("$", " ".join(command), flush=True)
    return subprocess.run(command, cwd=cwd, check=True)

In [ ]:
try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError("Connect this notebook to a Google Colab runtime first.") from exc

drive.mount(str(DRIVE_MOUNT), force_remount=False)

if (REPO_DIR / ".git").is_dir():
    remote_url = subprocess.check_output(
        ["git", "remote", "get-url", "origin"], cwd=REPO_DIR, text=True
    ).strip()
    if remote_url != REPO_URL:
        raise RuntimeError(f"{REPO_DIR} belongs to a different repository: {remote_url}")
    run_checked(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR)
    run_checked(["git", "switch", BRANCH], cwd=REPO_DIR)
    dirty = subprocess.check_output(
        ["git", "status", "--porcelain"], cwd=REPO_DIR, text=True
    ).strip().splitlines()
    if dirty:
        print("Local repository changes found:")
        for change in dirty:
            print(f"  {change}")
        print("Trying a safe fast-forward update. Git will stop before overwriting a local file.")
    run_checked(["git", "merge", "--ff-only", f"origin/{BRANCH}"], cwd=REPO_DIR)
elif REPO_DIR.exists():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository.")
else:
    run_checked(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO_DIR])

commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
print(f"Repository ready: {REPO_DIR}")
print(f"Branch: {BRANCH}")
print(f"Commit: {commit}")

## 2. Copy the teacher data onto the runtime disk

The ZIP keeps the repository’s required `data/raw/teacher` structure.

In [ ]:
if not DATA_ZIP.is_file():
    raise FileNotFoundError(f"Dataset archive not found: {DATA_ZIP}")

teacher_dir = REPO_DIR / "data/raw/teacher"
required_files = (
    teacher_dir / "train/styles_train.csv",
    teacher_dir / "test/styles_prediction.csv",
)
image_suffixes = {".jpg", ".jpeg"}

with zipfile.ZipFile(DATA_ZIP) as archive:
    names = archive.namelist()
    unsafe_names = [
        name for name in names if Path(name).is_absolute() or ".." in Path(name).parts
    ]
    if unsafe_names:
        raise RuntimeError("The dataset archive contains an unsafe path.")
    expected_images = sum(
        name.startswith("data/raw/teacher/")
        and Path(name).suffix.lower() in image_suffixes
        for name in names
    )
    if expected_images == 0:
        raise RuntimeError("The archive has no teacher images in the expected folder.")
    current_images = sum(
        path.suffix.lower() in image_suffixes for path in teacher_dir.rglob("*")
    )
    needs_extract = current_images != expected_images or not all(
        path.is_file() for path in required_files
    )
    if needs_extract:
        print(f"Extracting {expected_images:,} teacher images...", flush=True)
        archive.extractall(REPO_DIR)
    else:
        print("Teacher data is already extracted; skipping.")

actual_images = sum(
    path.suffix.lower() in image_suffixes for path in teacher_dir.rglob("*")
)
missing_files = [str(path) for path in required_files if not path.is_file()]
if actual_images != expected_images or missing_files:
    raise RuntimeError(
        f"Dataset check failed: expected {expected_images:,} images, found "
        f"{actual_images:,}; missing files: {missing_files}"
    )
print(f"Teacher data ready: {actual_images:,} images")

## 3. Resolve the accepted parents and verify E7

These checks build each architecture and run a zero-input shape check. They create no optimizer and take no training step.

In [ ]:
os.chdir(REPO_DIR)
os.environ["FASHION_PROJECT_ROOT"] = str(REPO_DIR)
source_dir = str(REPO_DIR / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)

for output_dir in (
    DRIVE_TASK_DIR / "experiments",
    DRIVE_TASK_DIR / "logs",
    DRIVE_TASK_DIR / "results",
):
    output_dir.mkdir(parents=True, exist_ok=True)

from fashion.train.task3_experiments import (
    check_task3_child_setup,
    latest_completed_baseline_parent_run_ids,
    latest_completed_usage_e2_parent_run_ids,
)

usage_parent_run_ids = latest_completed_usage_e2_parent_run_ids(
    output_root=DRIVE_TASK_DIR
)
gender_parent_run_ids = latest_completed_baseline_parent_run_ids(
    "gender", output_root=DRIVE_TASK_DIR
)
usage_e7_check = check_task3_child_setup(
    "usage_tinyconvnext18",
    parent_run_ids=usage_parent_run_ids,
    root=REPO_DIR,
    device_name="cuda",
)
gender_e7_check = check_task3_child_setup(
    "gender_tinyhrnet20",
    parent_run_ids=gender_parent_run_ids,
    root=REPO_DIR,
    device_name="cuda",
)

if usage_e7_check["model_family"] != "task3_tinyconvnext18":
    raise RuntimeError("Usage E7 architecture contract changed")
if usage_e7_check["parameter_count"] != 384_345:
    raise RuntimeError("Usage E7 parameter contract changed")
if usage_e7_check["architecture_macs"] != 95_297_616:
    raise RuntimeError("Usage E7 MAC contract changed")
usage_child = usage_e7_check["child"]
if usage_child["class_weight_beta"] != 0.999 or usage_child["class_weight_cap"] != 5.0:
    raise RuntimeError("Usage E7 must keep the accepted E2 class weights")
if gender_e7_check["model_family"] != "task3_tinyhrnet20":
    raise RuntimeError("Gender E7 architecture contract changed")
if gender_e7_check["parameter_count"] != 374_445:
    raise RuntimeError("Gender E7 parameter contract changed")
if gender_e7_check["architecture_macs"] != 104_064_700:
    raise RuntimeError("Gender E7 MAC contract changed")

print("GPU:", usage_e7_check["environment"]["gpu"])
print("Usage E2 parents: ", usage_parent_run_ids)
print("Gender E1 parents:", gender_parent_run_ids)
print("Usage E7 parameters/MACs:", usage_e7_check["parameter_count"], usage_e7_check["architecture_macs"])
print("Gender E7 parameters/MACs:", gender_e7_check["parameter_count"], gender_e7_check["architecture_macs"])
print("Optimizer steps during checks:", usage_e7_check["optimizer_steps"], gender_e7_check["optimizer_steps"])

## 4. Train Usage E7: TinyConvNeXt-18

This changes only the accepted E2 model architecture. It keeps E2 effective-number class weights and all other controls.

In [ ]:
from fashion.train.task3_experiments import (
    audit_completed_registry_rows,
    run_task3_child_cv,
)

usage_e7_result = run_task3_child_cv(
    "usage_tinyconvnext18",
    parent_run_ids=usage_parent_run_ids,
    folds=range(5),
    root=REPO_DIR,
    output_root=DRIVE_TASK_DIR,
    registry_path=DRIVE_REGISTRY,
    registry_mirrors=[REPO_DIR / "results/runs.csv"],
    device_name="cuda",
)
usage_e7_registry_audit = audit_completed_registry_rows(
    DRIVE_REGISTRY, usage_e7_result["fold_run_ids"]
)
{
    "metrics_path": usage_e7_result["metrics_path"],
    "macro_f1": usage_e7_result["metrics"]["macro_f1"],
    "macro_f1_without_home": usage_e7_result["metrics"]["macro_f1_without_home"],
    "registry_audit": usage_e7_registry_audit,
}

## 5. Train Gender E7: TinyHRNet-20

This changes only the accepted E1 model architecture. It keeps ordinary cross-entropy and all other E1 controls.

In [ ]:
gender_e7_result = run_task3_child_cv(
    "gender_tinyhrnet20",
    parent_run_ids=gender_parent_run_ids,
    folds=range(5),
    root=REPO_DIR,
    output_root=DRIVE_TASK_DIR,
    registry_path=DRIVE_REGISTRY,
    registry_mirrors=[REPO_DIR / "results/runs.csv"],
    device_name="cuda",
)
gender_e7_registry_audit = audit_completed_registry_rows(
    DRIVE_REGISTRY, gender_e7_result["fold_run_ids"]
)
{
    "metrics_path": gender_e7_result["metrics_path"],
    "macro_f1": gender_e7_result["metrics"]["macro_f1"],
    "registry_audit": gender_e7_registry_audit,
}

## 6. Confirm the saved E7 artifacts

This is a factual training summary only. The frozen gates are applied in the main Task 3 notebook.

In [ ]:
import pandas as pd

parent_metric_paths = {
    "usage": DRIVE_TASK_DIR / "experiments/t3_usage_e2_class_balanced_ce/usage/aggregate/metrics.json",
    "gender": DRIVE_TASK_DIR / "baseline/gender/aggregate/metrics.json",
}
children = {"usage": usage_e7_result, "gender": gender_e7_result}
comparison = []
for target, child in children.items():
    parent = json.loads(parent_metric_paths[target].read_text(encoding="utf-8"))
    comparison.append(
        {
            "target": target,
            "parent_macro_f1": parent["macro_f1"],
            "e7_macro_f1": child["metrics"]["macro_f1"],
            "macro_f1_change": child["metrics"]["macro_f1"] - parent["macro_f1"],
            "model_family": child["metrics"]["model_family"],
            "parameter_count": child["metrics"]["parameter_count"],
            "architecture_macs": child["metrics"]["architecture_macs"],
            "e7_metrics_path": child["metrics_path"],
        }
    )
pd.DataFrame(comparison)